# SymbioGPT-10M Teacher Training (Colab A100)

Chinchilla-optimal training of the multi-organelle SymbioGPT teacher model.

- **Model**: SymbioGPT-10M (4 organelles: Conv, Monarch, LongConv, Attention)
- **Data**: 266M curated tokens from philosophy corpus (BPE-2000)
- **Steps**: 27,000 (single pass, Chinchilla optimal for 11M params)
- **GPU**: A100 recommended

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup - Install dependencies and clone repo
!pip install -q wandb huggingface_hub
!git clone https://github.com/DavinciDreams/SymbioGPT.git /content/SymbioGPT
%cd /content/SymbioGPT

In [ ]:
# 2. Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# 3. Download pre-tokenized data from HuggingFace
import os
from huggingface_hub import hf_hub_download

os.makedirs("data", exist_ok=True)

HF_REPO = "LisaMegaWatts/SymbioTeacher-10M"

print("Downloading training tokens (1GB)...")
train_cache = hf_hub_download(
    repo_id=HF_REPO,
    filename="data/train_curated.txt.tokens.pt",
    local_dir=".",
)
print(f"Train tokens: {train_cache}")

print("Downloading validation tokens (277MB)...")
val_cache = hf_hub_download(
    repo_id=HF_REPO,
    filename="data/val.txt.tokens.pt",
    local_dir=".",
)
print(f"Val tokens: {val_cache}")

In [ ]:
# 4. W&B Login
import wandb
wandb.login()  # Will prompt for API key in Colab

In [ ]:
# 5. Load model and data
import sys
sys.path.insert(0, "/content/SymbioGPT")

from symbio_model import (
    SymbioConfig,
    SymbioGPT,
    complexity_penalty,
    compute_gate_entropy,
    compute_symbio_params,
)

config = SymbioConfig(
    d_model=320,
    n_layers=8,
    n_heads=5,
    head_dim=64,
    ffn_mult=4,
    context_length=256,
    vocab_size=2000,
    weight_tying=True,
    organelles=("causal_conv", "monarch", "long_conv", "attention"),
    conv_kernel_size=4,
    n_monarch_heads=1,
    gate_temperature_init=1.0,
    free_energy_beta=0.001,
)

total_params = compute_symbio_params(config)
print(f"SymbioGPT config: d={config.d_model}, L={config.n_layers}, params={total_params:,}")

# Load cached tokens
print("Loading cached tokens...")
train_tokens = torch.load("data/train_curated.txt.tokens.pt", weights_only=True).tolist()
val_tokens = torch.load("data/val.txt.tokens.pt", weights_only=True).tolist()
print(f"Train: {len(train_tokens):,} tokens")
print(f"Val: {len(val_tokens):,} tokens")

# Chunk into sequences
CTX = config.context_length
def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f"Train seqs: {len(train_inputs):,} ({len(train_inputs)*CTX:,} tokens)")
print(f"Val seqs: {len(val_inputs):,}")
del train_tokens, val_tokens  # free memory

In [ ]:
# 6. Training configuration
import math
import time
import torch.nn.functional as F

# Hyperparameters
BATCH_SIZE = 64          # A100 can handle larger batches
GRAD_ACCUM = 1           # No accumulation needed with larger batch
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
TOTAL_STEPS = 27_000     # Chinchilla optimal
BASE_LR = 6e-4
WARMUP_STEPS = 500
BETA = config.free_energy_beta
PRECISION = "bf16"       # A100 excels at BF16

CHECKPOINT_DIR = "checkpoints/teacher"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device("cuda")

# Model
model = SymbioGPT(config).to(device)
actual_params = sum(p.numel() for p in model.parameters())
print(f"Model on {device}: {actual_params:,} params")

# AMP
if PRECISION == "bf16" and torch.cuda.is_bf16_supported():
    amp_dtype = torch.bfloat16
    print("Using BF16 mixed precision")
elif PRECISION == "fp16":
    amp_dtype = torch.float16
    print("Using FP16 mixed precision")
else:
    amp_dtype = None
    print("Using FP32")

scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

# torch.compile
if hasattr(torch, "compile"):
    print("Compiling model with torch.compile...")
    model = torch.compile(model)
    print("Done")

# Optimizer + scheduler
optimizer = torch.optim.AdamW(
    model.parameters(), lr=BASE_LR, weight_decay=0.1, betas=(0.9, 0.95)
)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return (step + 1) / max(WARMUP_STEPS, 1)
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# W&B
run = wandb.init(
    project="symbiogenesis",
    name="symbio-teacher-10m-chinchilla",
    config={
        "architecture": "SymbioGPT",
        "d_model": config.d_model,
        "n_layers": config.n_layers,
        "n_heads": config.n_heads,
        "organelles": list(config.organelles),
        "total_params": total_params,
        "actual_params": actual_params,
        "batch_size": BATCH_SIZE,
        "effective_batch": EFFECTIVE_BATCH,
        "total_steps": TOTAL_STEPS,
        "base_lr": BASE_LR,
        "warmup_steps": WARMUP_STEPS,
        "precision": PRECISION,
        "free_energy_beta": BETA,
        "context_length": CTX,
        "vocab_size": config.vocab_size,
        "weight_tying": config.weight_tying,
        "hardware": "colab-a100",
    },
    tags=["teacher", "symbio", "10m", "chinchilla", "a100"],
)
print(f"W&B run: {run.url}")

tokens_per_step = EFFECTIVE_BATCH * CTX
total_train_tokens = TOTAL_STEPS * tokens_per_step
print(f"\nTraining plan:")
print(f"  Steps: {TOTAL_STEPS:,}")
print(f"  Tokens/step: {tokens_per_step:,}")
print(f"  Total tokens: {total_train_tokens:,}")
print(f"  Chinchilla ratio: {total_train_tokens/actual_params:.1f}x params")

In [ ]:
# 7. Evaluation function
def evaluate(model, val_inputs, val_labels, batch_size, device, amp_dtype):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for i in range(0, len(val_inputs), batch_size):
            batch_in = val_inputs[i:i+batch_size].to(device)
            batch_tgt = val_labels[i:i+batch_size].to(device)
            with torch.amp.autocast("cuda", enabled=amp_dtype is not None, dtype=amp_dtype):
                logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction="sum"
            )
            total_loss += loss.item()
            total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl

In [ ]:
# 8. Training loop
n_train = len(train_inputs)
step = 0
best_val_loss = float("inf")
t_start = time.time()

print(f"Starting training: {TOTAL_STEPS} steps, batch={BATCH_SIZE}, lr={BASE_LR:.1e}, beta={BETA}")
print(f"Train sequences: {n_train:,}, tokens/epoch: {n_train*CTX:,}")
print()

model.train()
accum_ce = 0.0
accum_fe = 0.0
micro_step = 0

while step < TOTAL_STEPS:
    perm = torch.randperm(n_train)
    for i in range(0, n_train, BATCH_SIZE):
        if step >= TOTAL_STEPS:
            break

        idx = perm[i:i+BATCH_SIZE]
        batch_in = train_inputs[idx].to(device)
        batch_tgt = train_labels[idx].to(device)

        with torch.amp.autocast("cuda", enabled=amp_dtype is not None, dtype=amp_dtype):
            logits = model(batch_in)
            B, T, V = logits.shape
            ce_loss = F.cross_entropy(logits.reshape(B*T, V), batch_tgt.reshape(B*T))
            if BETA > 0:
                fe_penalty = complexity_penalty(model)
                loss = (ce_loss + BETA * fe_penalty) / GRAD_ACCUM
            else:
                loss = ce_loss / GRAD_ACCUM
                fe_penalty = torch.tensor(0.0)

        scaler.scale(loss).backward()
        accum_ce += ce_loss.item()
        accum_fe += fe_penalty.item() if isinstance(fe_penalty, torch.Tensor) else 0.0
        micro_step += 1

        if micro_step < GRAD_ACCUM:
            continue

        # Optimizer step
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        avg_ce = accum_ce / GRAD_ACCUM
        avg_fe = accum_fe / GRAD_ACCUM
        accum_ce = 0.0
        accum_fe = 0.0
        micro_step = 0

        # Logging
        if step % 50 == 0:
            elapsed = time.time() - t_start
            tokens_per_sec = (step + 1) * EFFECTIVE_BATCH * CTX / max(elapsed, 1)
            gate_entropy = compute_gate_entropy(model)
            lr_now = scheduler.get_last_lr()[0]

            wandb.log({
                "train/ce_loss": avg_ce,
                "train/free_energy": avg_fe,
                "train/total_loss": avg_ce + BETA * avg_fe,
                "train/gate_entropy": gate_entropy,
                "train/grad_norm": grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm,
                "train/lr": lr_now,
                "train/tokens_per_sec": tokens_per_sec,
            }, step=step)

            if step % 500 == 0:
                print(
                    f"[step {step:5d}/{TOTAL_STEPS}] CE={avg_ce:.4f} FE={avg_fe:.4f} "
                    f"GateH={gate_entropy:.3f} LR={lr_now:.2e} tok/s={tokens_per_sec:.0f} "
                    f"elapsed={elapsed:.0f}s"
                )

        # Validation
        if step > 0 and step % 500 == 0:
            val_loss, val_ppl = evaluate(model, val_inputs, val_labels, BATCH_SIZE, device, amp_dtype)
            wandb.log({"val/loss": val_loss, "val/perplexity": val_ppl}, step=step)
            marker = " ** NEW BEST **" if val_loss < best_val_loss else ""
            print(f"[step {step:5d}] val_loss={val_loss:.4f} val_ppl={val_ppl:.1f}{marker}")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "symbio_best.pt"))
            model.train()

        # Checkpoint
        if step > 0 and step % 2000 == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"symbio_step{step}.pt")
            torch.save(model.state_dict(), ckpt_path)
            print(f"Checkpoint saved: {ckpt_path}")

        step += 1

print(f"\nTraining complete! {TOTAL_STEPS} steps in {time.time()-t_start:.0f}s")

In [ ]:
# 9. Final evaluation and save
val_loss, val_ppl = evaluate(model, val_inputs, val_labels, BATCH_SIZE, device, amp_dtype)
print(f"Final: val_loss={val_loss:.4f}, val_ppl={val_ppl:.1f}")
wandb.log({"val/final_loss": val_loss, "val/final_ppl": val_ppl})

# Save final model
final_path = os.path.join(CHECKPOINT_DIR, "symbio_final.pt")
torch.save(model.state_dict(), final_path)
print(f"Final model saved: {final_path}")

# Gate specialization report
print("\n=== Gate Specialization ===")
gate_weights = model.get_gate_weights()
organelle_names = list(config.organelles)
for i, w in enumerate(gate_weights):
    mean_w = w.mean(dim=1)
    for j, name in enumerate(organelle_names):
        wandb.summary[f"gate/layer{i}/{name}"] = mean_w[j].item()
    report = " | ".join(f"{name}={mean_w[j]:.3f}" for j, name in enumerate(organelle_names))
    print(f"  Layer {i}: {report}")

In [ ]:
# 10. Upload to HuggingFace
from huggingface_hub import HfApi, login
login()  # Will prompt for HF token

api = HfApi()
HF_REPO = "LisaMegaWatts/SymbioTeacher-10M"

# Upload final model
api.upload_file(
    path_or_fileobj=final_path,
    path_in_repo="symbio_final.pt",
    repo_id=HF_REPO,
    commit_message=f"Upload SymbioGPT-10M teacher (val_ppl={val_ppl:.1f}, {TOTAL_STEPS} steps, A100)",
)

# Upload best model
best_path = os.path.join(CHECKPOINT_DIR, "symbio_best.pt")
if os.path.exists(best_path):
    api.upload_file(
        path_or_fileobj=best_path,
        path_in_repo="symbio_best.pt",
        repo_id=HF_REPO,
        commit_message=f"Upload best checkpoint (val_loss={best_val_loss:.4f})",
    )

print(f"Uploaded to: https://huggingface.co/{HF_REPO}")

wandb.finish()
print("Done!")